# LSTM IDS — Colab Pipeline (GitHub Workflow)

This notebook pulls the latest code from GitHub, extracts pre-downloaded
raw datasets from Google Drive, and runs the full LSTM IDS pipeline on GPU.

**Workflow:** Git clone/pull → Install deps → Extract data → Run pipeline → Backup to Drive

**Features:**
- Auto-resume after Colab disconnect (uses `--resume` flag)
- Per-dataset output isolation (`outputs/<dataset>/`)
- Automatic backup to Google Drive after each dataset
- Environment validation (GPU, RAM, disk, Drive mount)
- Memory-optimized: disk-based splits, auto-sampling baselines

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Google Drive mounted at /content/drive')

## Cell 2 — Clone or Pull Project from GitHub

In [ ]:
import os
import subprocess

PROJECT_DIR = '/content/lstm_ids_project'
REPO_URL = 'https://github.com/Nico3783/lstm_ids_project.git'

# Try to get GitHub token from Colab Secrets (preferred)
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
    if GH_TOKEN:
        # Inject token into URL for private repo access
        AUTH_URL = REPO_URL.replace('https://', f'https://{GH_TOKEN}@')
        print('Using GitHub token from Colab Secrets.')
    else:
        AUTH_URL = REPO_URL
        print('No GH_TOKEN found. Using public URL (may fail for private repos).')
except Exception:
    AUTH_URL = REPO_URL
    print('Colab Secrets not available. Using public URL.')

if os.path.exists(PROJECT_DIR):
    # Pull latest changes
    print('Project exists. Pulling latest changes...')
    result = subprocess.run(
        ['git', '-C', PROJECT_DIR, 'pull', '--ff-only'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'git pull failed: {result.stderr}')
        print('Force-resetting to origin/main...')
        subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', PROJECT_DIR, 'reset', '--hard', 'origin/main'], check=True)
    print('Project updated.')
else:
    # Clone fresh
    print('Cloning project...')
    result = subprocess.run(
        ['git', 'clone', AUTH_URL, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f'git clone failed: {result.stderr}')
    print('Project cloned.')

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
!git log --oneline -3

## Cell 3 — Rename TensorFlow Shim & Install Dependencies

In [ ]:
import os

# Rename local tensorflow/ shim so real TF is used on Colab
shim = 'tensorflow'
shim_rename = 'tensorflow_shim_local'
if os.path.isdir(shim) and not os.path.isdir(shim_rename):
    os.rename(shim, shim_rename)
    print(f'Renamed {shim}/ → {shim_rename}/')
else:
    print('TF shim already renamed or not present.')

# Install dependencies
!pip install -q -r requirements_colab.txt
print('Dependencies installed.')

## Cell 4 — Extract Raw Datasets from Drive

In [ ]:
import zipfile
import os

DRIVE_ZIP = '/content/drive/MyDrive/lstm_raw.zip'
DATA_DIR = 'data/raw'

os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(DRIVE_ZIP):
    print(f'Extracting {DRIVE_ZIP} → {DATA_DIR}/...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    print('Extraction complete.')
else:
    print(f'WARNING: {DRIVE_ZIP} not found. Datasets may already be extracted.')

# Verify
for dset in ['NSL-KDD', 'CICIDS2017', 'UNSW-NB15']:
    path = os.path.join(DATA_DIR, dset)
    status = 'FOUND' if os.path.isdir(path) else 'MISSING'
    print(f'  {dset}: {status}')

## Cell 5 — Environment Validation (GPU, RAM, Disk, TensorFlow)

In [ ]:
import os, platform, subprocess

# Set Keras backend to TensorFlow before any imports
os.environ['KERAS_BACKEND'] = 'tensorflow'

# GPU check
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f'GPU detected: {gpus[0].name}')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print('GPU memory growth enabled.')
    else:
        print('WARNING: No GPU detected. Training will be slow on CPU.')
except Exception as e:
    print(f'TensorFlow GPU check failed: {e}')

# RAM check
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f'RAM: {ram_gb:.1f} GB')
    if ram_gb < 12:
        print('WARNING: Less than 12 GB RAM. Large datasets may cause OOM.')
except ImportError:
    print('psutil not available — skipping RAM check.')

# Disk check
st = os.statvfs('.')
disk_gb = (st.f_bavail * st.f_frsize) / (1024**3)
print(f'Disk free: {disk_gb:.1f} GB')

# Drive mount check
drive_ok = os.path.isdir('/content/drive/MyDrive')
print(f'Drive mount: {"OK" if drive_ok else "MISSING"}')

# TensorFlow version
print(f'TensorFlow: {tf.__version__}')
print(f'Python: {platform.python_version()}')
print('Environment validation complete.')

## Cell 6 — Run NSL-KDD Pipeline

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

# Rename TF shim (idempotent)
if os.path.isdir('tensorflow') and not os.path.isdir('tensorflow_shim_local'):
    os.rename('tensorflow', 'tensorflow_shim_local')

!python run_pipeline.py --dataset nsl_kdd --resume --epochs 100 --batch-size 64 --log-level INFO

## Cell 7 — Backup NSL-KDD Results to Drive

In [ ]:
import shutil, os

BACKUP_BASE = '/content/drive/MyDrive/lstm_ids_results'
os.makedirs(BACKUP_BASE, exist_ok=True)

src = 'outputs/nsl_kdd'
dst = os.path.join(BACKUP_BASE, 'nsl_kdd')

if os.path.isdir(src):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Backed up NSL-KDD → {dst}')
else:
    print(f'WARNING: {src} not found. Pipeline may have failed.')

## Cell 8 — Run UNSW-NB15 Pipeline

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

# Rename TF shim (idempotent)
if os.path.isdir('tensorflow') and not os.path.isdir('tensorflow_shim_local'):
    os.rename('tensorflow', 'tensorflow_shim_local')

!python run_pipeline.py --dataset unsw_nb15 --resume --epochs 100 --batch-size 64 --subsample 0.3 --log-level INFO

## Cell 9 — Backup UNSW-NB15 Results to Drive

In [ ]:
import shutil, os

BACKUP_BASE = '/content/drive/MyDrive/lstm_ids_results'
os.makedirs(BACKUP_BASE, exist_ok=True)

src = 'outputs/unsw_nb15'
dst = os.path.join(BACKUP_BASE, 'unsw_nb15')

if os.path.isdir(src):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Backed up UNSW-NB15 → {dst}')
else:
    print(f'WARNING: {src} not found.')

## Cell 10 — Run CICIDS2017 Pipeline

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

# Rename TF shim (idempotent)
if os.path.isdir('tensorflow') and not os.path.isdir('tensorflow_shim_local'):
    os.rename('tensorflow', 'tensorflow_shim_local')

!python run_pipeline.py --dataset cicids2017 --resume --epochs 100 --batch-size 64 --log-level INFO

## Cell 11 — Backup CICIDS2017 Results to Drive

In [ ]:
import shutil, os

BACKUP_BASE = '/content/drive/MyDrive/lstm_ids_results'
os.makedirs(BACKUP_BASE, exist_ok=True)

src = 'outputs/cicids2017'
dst = os.path.join(BACKUP_BASE, 'cicids2017')

if os.path.isdir(src):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Backed up CICIDS2017 → {dst}')
else:
    print(f'WARNING: {src} not found.')

## Cell 12 — Download All Results as ZIP

In [ ]:
import shutil, os

BACKUP_BASE = '/content/drive/MyDrive/lstm_ids_results'

# Also copy from local outputs if Drive backup doesn't exist yet
for dset in ['nsl_kdd', 'unsw_nb15', 'cicids2017']:
    local = f'outputs/{dset}'
    drive = os.path.join(BACKUP_BASE, dset)
    if os.path.isdir(local) and not os.path.isdir(drive):
        shutil.copytree(local, drive)
        print(f'Copied {local} → {drive}')

# Create ZIP for download
if os.path.isdir(BACKUP_BASE):
    shutil.make_archive('/content/lstm_ids_results', 'zip', BACKUP_BASE)
    print('\nZIP created: /content/lstm_ids_results.zip')
    from google.colab import files
    files.download('/content/lstm_ids_results.zip')
else:
    print('No results found to download.')